# VieNeu-TTS Fine-tuning Notebook

Notebook này tổng hợp toàn bộ code training cho **VieNeu-TTS-0.3B**.  

## 📦 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q transformers peft torch datasets librosa soundfile tqdm phonemizer
!pip install -q git+https://github.com/Neuphonic/NeuCodec.git

## 🔧 2. Setup Utils 

In [ ]:
!pip install sea-g2p -q

In [ ]:
import sys
from sea_g2p import Normalizer, G2P
def setup_text_engine():
    """Thiết lập engine Phonemize & Normalize bằng sea-g2p"""
    
    # Khởi tạo các engine
    norm = Normalizer()
    g2p_engine = G2P()
    
    # Wrapper để giữ nguyên API cũ cho các cell code phía sau
    def normalize_text(text):
        return norm.normalize(text)
    
    def phonemize_with_dict(text):
        # Chuyển văn bản sang âm vị IPA
        return g2p_engine.convert(text)
    
    print("✅ Text Engine loaded via sea-g2p (Pure Python)")
    return normalize_text, phonemize_with_dict
# Khởi chạy thiết lập
normalize_text, phonemize_with_dict = setup_text_engine()

# ========== QUY TRÌNH TEST ==========
def preprocess_text(text):
    """Quy trình tiền xử lý hoàn chỉnh"""
    normalized = normalize_text(text)
    phonemes = phonemize_with_dict(normalized)
    return {
        "original": text,
        "normalized": normalized,
        "phonemes": phonemes
    }
# Chạy thử nghiệm
result = preprocess_text("Tôi có 2.000 mẫu audio, giá 5.000.000đ")
print(f"\n📝 Kết quả thực tế từ sea-g2p:")
for key, val in result.items():
    print(f"  {key:12s}: {val}")
print("\n🎉 Hệ thống tiền xử lý đã sẵn sàng!")

## 📥 3. Download Sample Data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aimy144/trainxttsv2-audiobook")

print("Path to dataset files:", path)

!cp -r {path}/* ./dataset/ -v

In [ ]:
import pandas as pd

df1 = pd.read_csv("dataset/metadata_train.csv", sep="|")
df2 = pd.read_csv("dataset/metadata_eval.csv", sep="|")

merged = pd.concat([df1, df2], ignore_index=True)

merged.to_csv("dataset/metadata.csv", sep="|", index=False)

print(merged.head())
print(merged.shape)

In [ ]:
import pandas as pd

input_csv = "dataset/metadata.csv"
output_csv = "dataset/metadata.csv"

# File không có header
df = pd.read_csv(
    input_csv,
    sep="|",
    names=["file_name", "text"],
    encoding="utf-8"
)

print(df.head())

df.to_csv(
    output_csv,
    sep="|",
    index=False,
    header=False,
    encoding="utf-8"
)

print("Done")
print(df.head())

## 🧹 4. Filter Data

Lọc dữ liệu kém chất lượng (audio hỏng, text rác, quá ngắn/dài)

In [ ]:
import io
import os
from datasets import load_dataset, Audio
import soundfile as sf
from tqdm import tqdm
import re
ACRONYM = re.compile(r"(?:[a-zA-Z]\.){2,}")
ACRONYM_NO_PERIOD = re.compile(r"(?:[A-Z]){2,}")

def text_filter(text: str) -> bool:
    if not text: return False
    if re.search(r"\d", text): return False
    if ACRONYM.search(text) or ACRONYM_NO_PERIOD.search(text): return False
    if text[-1] not in ".,?!": return False
    return True

def filter_dataset(dataset_dir="dataset"):
    metadata_path = os.path.join(dataset_dir, "metadata.csv")
    cleaned_path = os.path.join(dataset_dir, "metadata_cleaned.csv")
    raw_audio_dir = os.path.join(dataset_dir)
    
    if not os.path.exists(metadata_path):
        print(f"❌ Không tìm thấy {metadata_path}")
        return
    
    print("🧹 Bắt đầu lọc dữ liệu...")
    
    valid_samples = []
    skipped = {"audio_not_found": 0, "audio_error": 0, "duration_out_of_range": 0, "text_invalid": 0}
    
    with open(metadata_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    for line in tqdm(lines, desc="Filtering"):
        parts = line.strip().split('|')
        if len(parts) < 2:
            continue
        
        filename = parts[0]
        text = parts[1]
        file_path = os.path.join(raw_audio_dir, filename)
        
        if not os.path.exists(file_path):
            skipped["audio_not_found"] += 1
            continue
        
        try:
            info = sf.info(file_path)
            duration = info.duration
            
            if not (3.0 <= duration <= 15.0):
                skipped["duration_out_of_range"] += 1
                continue
        except Exception:
            skipped["audio_error"] += 1
            continue
        
        if not text_filter(text):
            skipped["text_invalid"] += 1
            continue
        
        valid_samples.append(f"{filename}|{text}\n")
    
    with open(cleaned_path, 'w', encoding='utf-8') as f:
        f.writelines(valid_samples)
    
    print(f"\n🦜 KẾT QUẢ LỌC:")
    print(f"   - Tổng: {len(lines)} | Hợp lệ: {len(valid_samples)} ({len(valid_samples)/len(lines)*100:.1f}%)")
    print(f"   - Loại bỏ: {sum(skipped.values())} ({skipped})")
    print(f"✅ Đã lưu: {cleaned_path}")
    return cleaned_path

cleaned_metadata_path = filter_dataset(dataset_dir="dataset")

In [ ]:
import pandas as pd

df = pd.read_csv(
    "dataset/metadata_cleaned.csv",
    sep="|",
    encoding="utf-8"
)

print(df.head())

## 🔊 5. Encode Audio to VQ Codes

Sử dụng NeuCodec để encode audio thành vector quantized codes

In [ ]:
import torch
import librosa
from neucodec import NeuCodec
import json
import random

def encode_dataset(dataset_dir="dataset", max_samples=2000):
    metadata_path = os.path.join(dataset_dir, "metadata_cleaned.csv")
    if not os.path.exists(metadata_path):
        print(f"🦜 Không tìm thấy metadata_cleaned.csv, dùng metadata.csv...")
        metadata_path = os.path.join(dataset_dir, "metadata.csv")
    
    output_path = os.path.join(dataset_dir, "metadata_encoded.csv")
    raw_audio_dir = os.path.join(dataset_dir)
    
    if not os.path.exists(metadata_path):
        print("🦜 Không tìm thấy metadata!")
        return
    
    print("🦜 Đang tải NeuCodec model...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    codec = NeuCodec.from_pretrained("neuphonic/neucodec").to(device)
    codec.eval()
    
    print(f"🦜 Encode tối đa {max_samples} mẫu (device: {device})")
    
    lines_to_write = []
    skipped_count = 0
    
    with open(metadata_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Shuffle và lấy max_samples
    random.shuffle(lines)
    if len(lines) > max_samples:
        lines = lines[:max_samples]
    
    for line in tqdm(lines, desc="Encoding"):
        parts = line.strip().split('|')
        if len(parts) < 2:
            continue
        
        filename = parts[0]
        text = parts[1]
        audio_path = os.path.join(raw_audio_dir, filename)
        
        if not os.path.exists(audio_path):
            skipped_count += 1
            print(audio_path)
            continue
        
        try:
            wav, sr = librosa.load(audio_path, sr=16000, mono=True)
            wav_tensor = torch.from_numpy(wav).float().unsqueeze(0).unsqueeze(0)
            
            with torch.no_grad():
                codes = codec.encode_code(wav_tensor)
                codes = codes.squeeze(0).squeeze(0).cpu().numpy().flatten().tolist()
                codes = [int(x) for x in codes]
            
            # Validate
            if not codes or not all(0 <= c < 65536 for c in codes):
                print(f"🦜 Invalid codes: {filename}")
                skipped_count += 1
                continue
            
            codes_json = json.dumps(codes)
            lines_to_write.append(f"{filename}|{text}|{codes_json}\n")
            
        except Exception as e:
            print(f"🦜 Lỗi {filename}: {e}")
            skipped_count += 1
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.writelines(lines_to_write)
    
    print(f"\n🦜 Hoàn tất! Đã encode {len(lines_to_write)} mẫu")
    print(f"   - Lưu tại: {output_path}")
    print(f"   - Bỏ qua: {skipped_count}")
    return output_path

encoded_metadata_path = encode_dataset(dataset_dir="dataset", max_samples=4000)

## 🎯 6. Setup Training

Cấu hình LoRA và Training Arguments

In [ ]:
!rm -r output
!rm -r vieneu_merged_lora

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import TrainingArguments

# LoRA Config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Training Config
training_config = {
    'model': "pnnbao-ump/VieNeu-TTS-0.3B",
    'run_name': "VieNeu-TTS-LoRA",
    'output_dir': "output",
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 1,
    'learning_rate': 2e-4,
    'max_steps': 5000,
    'logging_steps': 50,
    'save_steps': 500,
    'eval_steps': 500,
    'warmup_ratio': 0.05,
    'bf16': True,
}

def get_training_args(config):
    return TrainingArguments(
        output_dir=os.path.join(config['output_dir'], config['run_name']),
        do_train=True,
        do_eval=True,
        max_steps=config['max_steps'],
        per_device_train_batch_size=config['per_device_train_batch_size'],
        gradient_accumulation_steps=config['gradient_accumulation_steps'],
        learning_rate=config['learning_rate'],
        warmup_ratio=config['warmup_ratio'],
        bf16=config['bf16'],
        logging_steps=config['logging_steps'],
        save_steps=config['save_steps'],
        eval_strategy="steps",
        eval_steps=config['eval_steps'],
        save_strategy="steps",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        dataloader_num_workers=2,
        ddp_find_unused_parameters=False,
    )

print("✅ Training config ready!")

## 📊 7. Dataset Class & Preprocessing

In [ ]:
from torch.utils.data import Dataset

def preprocess_sample(sample, tokenizer, max_len=2048):
    speech_gen_start = tokenizer.convert_tokens_to_ids('<|SPEECH_GENERATION_START|>')
    ignore_index = -100
    
    phones = sample["phones"]
    vq_codes = sample["codes"]
    
    codes_str = "".join([f"<|speech_{i}|>" for i in vq_codes])
    chat = f"""<|TEXT_PROMPT_START|>{phones}<|TEXT_PROMPT_END|><|SPEECH_GENERATION_START|>{codes_str}<|SPEECH_GENERATION_END|>"""
    
    ids = tokenizer.encode(chat)
    
    # Pad/truncate
    if len(ids) < max_len:
        ids = ids + [tokenizer.pad_token_id] * (max_len - len(ids))
    elif len(ids) > max_len:
        ids = ids[:max_len]
    
    input_ids = torch.tensor(ids, dtype=torch.long)
    labels = torch.full_like(input_ids, ignore_index)
    
    # Mask labels before speech generation
    speech_gen_start_idx = (input_ids == speech_gen_start).nonzero(as_tuple=True)[0]
    if len(speech_gen_start_idx) > 0:
        speech_gen_start_idx = speech_gen_start_idx[0]
        labels[speech_gen_start_idx:] = input_ids[speech_gen_start_idx:]
    
    attention_mask = (input_ids != tokenizer.pad_token_id).long()
    
    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }

class VieNeuDataset(Dataset):
    def __init__(self, metadata_path, tokenizer, max_len=2048):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        if not os.path.exists(metadata_path):
            raise FileNotFoundError(f"Missing: {metadata_path}")
        
        with open(metadata_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('|')
                if len(parts) >= 3:
                    self.samples.append({
                        "filename": parts[0],
                        "text": parts[1],
                        "codes": json.loads(parts[2])
                    })
        
        print(f"🦜 Loaded {len(self.samples)} samples from {metadata_path}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        text = sample["text"]
        
        try:
            phones = phonemize_with_dict(text)
        except Exception as e:
            print(f"⚠️ Phonemization error: {e}")
            phones = text
        
        data_item = {"phones": phones, "codes": sample["codes"]}
        return preprocess_sample(data_item, self.tokenizer, self.max_len)

print("✅ Dataset class ready!")

## 🚀 8. Train Model

In [ ]:
import gc
import torch

for name in ["model", "base_model", "codec", "trainer"]:
    if name in globals():
        del globals()[name]
        print(f"Deleted: {name}")

gc.collect()

if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("✅ CUDA cache cleaned")
    except Exception as e:
        print("⚠️ CUDA context already crashed. Please restart session.")
        print(e)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, default_data_collator

model_name = training_config['model']
print(f"🦜 Loading model: {model_name}")

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

# Load Dataset
dataset_path = encoded_metadata_path
full_dataset = VieNeuDataset(dataset_path, tokenizer)

# Train/Eval split (5%)
val_size = max(1, int(0.05 * len(full_dataset)))
train_size = len(full_dataset) - val_size
train_dataset, eval_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

print(f"🦜 Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# Apply LoRA
print("🦜 Applying LoRA...")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Trainer
args = get_training_args(training_config)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
)

print("🦜 Starting training! (Good luck)")
trainer.train()

# Save
save_path = os.path.join(training_config['output_dir'], training_config['run_name'])
print(f"🦜 Saving model to: {save_path}")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Training complete!")

## 🦜 Done!

Model đã được fine-tune và lưu tại `output/VieNeu-TTS-LoRA/`.

Bạn có thể sử dụng checkpoint này để:
- Inference / generate speech
- Merge LoRA vào model gốc
- Tiếp tục fine-tuning với dataset khác